In [7]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 1. 加载系统环境变量
load_dotenv()

# 2. 默认初始化模型
model = init_chat_model(
    # 自动推断base_url、模型供应商的名称、去环境变量中自动加载对应的API_KEY
    model="deepseek-v4-flash",
    temperature=0.7,
    # 结构化输出,必须关闭思考模式
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

In [11]:
# 4.阻塞式多轮调用智能体
from langchain_core.messages import AIMessage
response:AIMessage = model.invoke([
    {"role":"system", "content":"你是一个脱口秀演员,用轻松幽默的方式回答问题"},
    {"role":"user","content":"我现在在家,要去洗车,家里距离洗车店100米,我要开车去吗?"},
    {"role":"assistant", "content":"这是对我的考验呀!"},
    {"role":"user","content":"请回答"},
])
response.pretty_print()

================================== Ai Message ==================================

哈哈，100米的距离开车去洗车？这位朋友，你是我见过最需要洗车的人——因为你的车可能已经停到积灰长蘑菇了，得赶紧开出去透透气！

不过说实话哈，100米……你要真开车去，洗车小哥估计得愣一下："哥，您是刚洗完回来还是专门来遛我的？"100米走路也就一分多钟，水温还没热呢就到了。

所以我给你个方案：你要是就想体验"车接车送"的仪式感，那就开去，反正油钱比洗车费便宜；你要想省事儿，直接跟洗车店说"我就住隔壁"，临走前让他们送你一张优惠券——记得说是朋友教的，别出卖我这个单口相声演员。 😄


In [5]:
# 4. 简化AIMessage
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
response:AIMessage = model.invoke([
    SystemMessage(content="你是一个脱口秀演员,用轻松幽默的方式回答问题"),
    HumanMessage(content="十个苹果三个人,苹果应该怎么分"),
    AIMessage(content="这是对我的考验呀!")
])
response.pretty_print()

================================== Ai Message ==================================

好的，让我们来认真分析一下这个"苹果分配"问题。三个人分十个苹果，最公平的方式当然是每人三个，剩下一个……嗯，这个剩下的苹果就成了关键！你可以把它切成三份，每人再分三分之一，完美解决！当然，如果你不想切苹果，也可以让一个人多拿一个，然后他负责下次买水果——这叫"投资未来"嘛！


In [10]:
# 为模型绑定固定的输出结构
from pydantic import BaseModel, Field

class CapitalInfo(BaseModel):
    name: str = Field(description="首都的名字")
    location: str = Field(description="首都的地址")
    vibe: str = Field(description="首都的氛围")
    economy: str = Field(description="首都的经济来源")

# 绑定结构化输出
model_with_structured_output = model.with_structured_output(schema=CapitalInfo)

# 接收返回的结构化数据
response:CapitalInfo = model_with_structured_output.invoke("你是一个小说家，你为我生成一个火星上的城市背景~")

# 以json的格式化输出
print(type(response))
print(response.model_dump_json(indent=4))

<class '__main__.CapitalInfo'>
{
    "name": "赤霄（Crimson Vault / 赤穹城）",
    "location": "火星阿尔西亚山区东麓的一处巨型熔岩管之内，入口面对深赤色的混沌地形（Chaos terrain）。整座城市躲在一公里厚的玄武岩顶板之下，冬季管口挂着结的干冰霜帘；城内第一层空间顶部有人造穹幕，播放着略带橘红的\"天空\"。三条地下磁悬浮城轨贯穿\"内城\"\"锈街\"\"底层井\"三区。",
    "vibe": "地底避难式的繁华，加上拓荒者的不驯。白天的街道靠荧光菜棚和小型熔盐堆的余光照明，灯一律偏远星橘色，因此全城仿佛永远处在黄昏。穹幕下面是有节律的社区合唱会和非正式的市集。地面雾浓说明高温和低压交汇形成了腐蚀性的微尘暴雨。市民对自己选票造成的损失气他们那种近乎迷信的概率统计，靠统计数据求仙转运的精神主义者。",
    "economy": "主经济：稀有金属（在山麓盐气凝胶中浮出一面的冷氢胞采矿与镍铼合金冶炼，产出火星最稀缺的砷。）第二：体水分净化（埋地下冻土的局切片工程、蒸发和蒸馏出的人造水站）。三地跨信用联：熔岩内外三层围叠的中大气拱工业内体去承（全暖气能微型化结构件的集中手图资本），但于邻区向外的大量专利池产生的管线热底工程站中处理塔。"
}
